# LeetCode 3898 — Find the Degree of Each Vertex bằng Python

Link đề: https://leetcode.com/problems/find-the-degree-of-each-vertex/ · Độ khó: **Easy** · Chủ đề: Array, Matrix, Graph

Skeleton LeetCode:

```python
class Solution:
    def findDegrees(self, matrix: List[List[int]]) -> List[int]:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org (Weekly Contest 497, câu 1, Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **ma trận kề (adjacency matrix)** của đồ thị vô hướng và khái niệm **bậc (degree)** của đỉnh: số cạnh nối với nó.
2. Nhận ra bậc của đỉnh `i` chính là **tổng của hàng `i`** (và cũng bằng tổng của cột `i` vì ma trận đối xứng).
3. Biết duyệt ma trận bằng hai vòng `for`, `sum(row)`, `zip(*matrix)` để lấy cột, `map`, và duyệt **nửa tam giác trên** để liệt kê cạnh.
4. Viết 5 cách khác nhau và kiểm chứng bằng **bổ đề bắt tay**: tổng các bậc bằng `2 × số cạnh`; test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: tính tổng cả ma trận, đếm số phần tử thay vì tổng, nhầm số cạnh với tổng bậc, cộng một đầu mút hai lần khi duyệt tam giác trên.

## 2. Đề bài & ví dụ

Cho ma trận `matrix` kích thước `n x n` là **ma trận kề** của một đồ thị **vô hướng** có `n` đỉnh đánh số `0..n-1`. `matrix[i][j] = 1` nghĩa là có cạnh nối đỉnh `i` và đỉnh `j`, `0` nghĩa là không có cạnh. Hãy trả về mảng `ans` độ dài `n` với `ans[i]` là **bậc** của đỉnh `i` (số cạnh nối với đỉnh `i`).

| Ví dụ | matrix | Output | Giải thích |
|---|---|---|---|
| 1 | `[[0,1,1],[1,0,1],[1,1,0]]` | `[2,2,2]` | Tam giác: mỗi đỉnh nối với hai đỉnh còn lại |
| 2 | `[[0,1,0],[1,0,0],[0,0,0]]` | `[1,1,0]` | Đỉnh 0 và 1 nối nhau; đỉnh 2 cô lập |
| 3 | `[[0]]` | `[0]` | Một đỉnh, không có cạnh |

Constraints: `1 <= n == matrix.length == matrix[i].length <= 100`; `matrix[i][i] == 0` (không có khuyên); mỗi `matrix[i][j]` là `0` hoặc `1`; `matrix[i][j] == matrix[j][i]` (đối xứng).

In [1]:
for m in ([[0, 1, 1], [1, 0, 1], [1, 1, 0]], [[0, 1, 0], [1, 0, 0], [0, 0, 0]], [[0]]):
    for row in m:
        print(row, "-> tổng hàng", sum(row))
    print()

[0, 1, 1] -> tổng hàng 2
[1, 0, 1] -> tổng hàng 2
[1, 1, 0] -> tổng hàng 2

[0, 1, 0] -> tổng hàng 1
[1, 0, 0] -> tổng hàng 1
[0, 0, 0] -> tổng hàng 0

[0] -> tổng hàng 0



## 3. Kiến thức nền

### 3.1. Đồ thị, đỉnh, cạnh và bậc

Đồ thị gồm **đỉnh** (nút) và **cạnh** (đoạn nối hai đỉnh). Đồ thị **vô hướng**: cạnh `i - j` là cạnh `j - i`. **Bậc** của một đỉnh là số cạnh gắn với nó. Đỉnh cô lập có bậc `0`.

In [2]:
edges = [(0, 1), (0, 2)]
deg = [0] * 3
for a, b in edges:
    deg[a] += 1
    deg[b] += 1
print(deg)         # đỉnh 0 nối với 1, 2 -> bậc 2

[2, 1, 1]


### 3.2. Ma trận kề

Ma trận kề `M` có `M[i][j] = 1` khi có cạnh giữa `i` và `j`. Với đồ thị vô hướng không khuyên: `M[i][j] == M[j][i]` (đối xứng) và đường chéo `M[i][i] = 0`. Hàng `i` liệt kê "đỉnh `i` nối với những đỉnh nào" bằng các số `1`.

In [3]:
M = [[0, 1, 0],
     [1, 0, 0],
     [0, 0, 0]]
n = len(M)
print(all(M[i][j] == M[j][i] for i in range(n) for j in range(n)), [M[i][i] for i in range(n)])
print("đỉnh 0 nối với:", [j for j in range(n) if M[0][j] == 1])

True [0, 0, 0]
đỉnh 0 nối với: [1]


### 3.3. `sum`, `map` và duyệt hàng

`sum(row)` cộng các số trong một hàng. `list(map(sum, matrix))` áp dụng `sum` cho **mỗi hàng** của ma trận, gọn như list comprehension `[sum(row) for row in matrix]`.

In [4]:
M = [[0, 1, 1], [1, 0, 1], [1, 1, 0]]
print([sum(row) for row in M], list(map(sum, M)))

[2, 2, 2] [2, 2, 2]


### 3.4. `zip(*matrix)` — chuyển vị (lấy cột)

`zip(*matrix)` ghép các phần tử cùng vị trí của các hàng, tức cho từng **cột**. Với ma trận đối xứng, tổng cột `j` bằng tổng hàng `j`.

In [5]:
M = [[0, 1, 0], [1, 0, 0], [0, 0, 0]]
print(list(zip(*M)))
print([sum(col) for col in zip(*M)], [sum(row) for row in M])

[(0, 1, 0), (1, 0, 0), (0, 0, 0)]
[1, 1, 0] [1, 1, 0]


### 3.5. Duyệt tam giác trên `i < j` và bổ đề bắt tay

Mỗi cạnh vô hướng xuất hiện **hai lần** trong ma trận (`M[i][j]` và `M[j][i]`). Duyệt chỉ `i < j` liệt kê mỗi cạnh một lần. **Bổ đề bắt tay:** tổng bậc của mọi đỉnh `= 2 × số cạnh`, nên tổng các bậc luôn **chẵn**.

In [6]:
M = [[0, 1, 1], [1, 0, 1], [1, 1, 0]]
n = len(M)
edges = [(i, j) for i in range(n) for j in range(i + 1, n) if M[i][j]]
print(edges, "số cạnh =", len(edges))
print("tổng bậc =", sum(map(sum, M)), "= 2 x", len(edges))

[(0, 1), (0, 2), (1, 2)] số cạnh = 3
tổng bậc = 6 = 2 x 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hàng `i` của ma trận kề nói "đỉnh `i` nối với đỉnh nào": mỗi số `1` ở `M[i][j]` là một cạnh nối `i` với `j`. Vì vậy:

> **bậc của đỉnh `i` = số số `1` trong hàng `i` = tổng hàng `i`** (vì mỗi phần tử chỉ là `0` hoặc `1`).

Tính tay ví dụ 2 `[[0,1,0],[1,0,0],[0,0,0]]`:

| đỉnh | hàng | tổng | bậc |
|---|---|---|---|
| 0 | `[0,1,0]` | 1 | 1 |
| 1 | `[1,0,0]` | 1 | 1 |
| 2 | `[0,0,0]` | 0 | 0 |

Kết quả `[1, 1, 0]`. Nhờ tính **đối xứng**, nếu tính theo cột cũng ra cùng kết quả; và vì đường chéo toàn `0`, không có cạnh nào bị tự đếm.

In [7]:
def degrees_by_hand(M):
    n = len(M)
    for i in range(n):
        neighbors = [j for j in range(n) if M[i][j] == 1]
        print(f"đỉnh {i}: hàng {M[i]} -> láng giềng {neighbors} -> bậc {len(neighbors)}")
    return [sum(row) for row in M]

print(degrees_by_hand([[0, 1, 0], [1, 0, 0], [0, 0, 0]]))
print(degrees_by_hand([[0, 1, 1], [1, 0, 1], [1, 1, 0]]))

đỉnh 0: hàng [0, 1, 0] -> láng giềng [1] -> bậc 1
đỉnh 1: hàng [1, 0, 0] -> láng giềng [0] -> bậc 1
đỉnh 2: hàng [0, 0, 0] -> láng giềng [] -> bậc 0
[1, 1, 0]
đỉnh 0: hàng [0, 1, 1] -> láng giềng [1, 2] -> bậc 2
đỉnh 1: hàng [1, 0, 1] -> láng giềng [0, 2] -> bậc 2
đỉnh 2: hàng [1, 1, 0] -> láng giềng [0, 1] -> bậc 2
[2, 2, 2]


## 5. Cách 1 — Brute Force / baseline

Hai vòng `for` lồng nhau: với mỗi đỉnh `i`, duyệt mọi `j` và đếm số lần `matrix[i][j] == 1`. Đúng định nghĩa "đếm số cạnh nối với `i`".

Đây đã là `O(n²)` — kích thước của đầu vào, nên không thể tốt hơn về bậc. "Nút thắt" chỉ là sự rườm rà của vòng lặp; các cách sau diễn đạt cùng ý gọn hơn (`sum`, `map`) hoặc từ góc nhìn khác (cột, danh sách cạnh).

In [8]:
class SolutionBrute:
    def findDegrees(self, matrix):
        n = len(matrix)
        ans = []
        for i in range(n):
            count = 0
            for j in range(n):
                if matrix[i][j] == 1:
                    count += 1
            ans.append(count)
        return ans

print(SolutionBrute().findDegrees([[0, 1, 1], [1, 0, 1], [1, 1, 0]]),
      SolutionBrute().findDegrees([[0, 1, 0], [1, 0, 0], [0, 0, 0]]),
      SolutionBrute().findDegrees([[0]]))

[2, 2, 2] [1, 1, 0] [0]


**Độ phức tạp:** Time `O(n²)`, Space `O(n)` cho mảng kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** vì mỗi phần tử là `0/1`, "đếm số `1`" chính là **cộng** — nên bậc của đỉnh `i` là `sum(matrix[i])`. Không cần `if`, không cần biến đếm.

**Invariant:** `ans[i] = sum(matrix[i])` luôn bằng số láng giềng của `i`, vì `matrix[i][j] = 1` đúng khi `j` là láng giềng của `i` (và `matrix[i][i] = 0` nên `i` không tự là láng giềng của mình).

Ba góc nhìn tương đương: tổng **hàng** (bằng `sum` hoặc `map(sum, ...)`), tổng **cột** (nhờ đối xứng), hoặc **đếm đầu mút của các cạnh** (duyệt tam giác trên). Bổ đề bắt tay cho một phép kiểm tra chéo: tổng các bậc phải chẵn và bằng `2E`.

## 7. Các cách tiếp theo

### Cách 2 — `sum` từng hàng (list comprehension)

`[sum(row) for row in matrix]` — một dòng.

In [9]:
class SolutionSumRow:
    def findDegrees(self, matrix):
        return [sum(row) for row in matrix]

print(SolutionSumRow().findDegrees([[0, 1, 1], [1, 0, 1], [1, 1, 0]]), SolutionSumRow().findDegrees([[0]]))

[2, 2, 2] [0]


**Độ phức tạp:** Time `O(n²)`, Space `O(n)`.

### Cách 3 — Tổng theo **cột** bằng `zip(*matrix)`

Nhờ ma trận đối xứng, tổng cột `j` bằng tổng hàng `j` bằng bậc của đỉnh `j`. Cách này cho thấy tính đối xứng; nó sai nếu ma trận không đối xứng (đồ thị có hướng).

In [10]:
class SolutionSumCol:
    def findDegrees(self, matrix):
        return [sum(col) for col in zip(*matrix)]

print(SolutionSumCol().findDegrees([[0, 1, 1], [1, 0, 1], [1, 1, 0]]), SolutionSumCol().findDegrees([[0, 1, 0], [1, 0, 0], [0, 0, 0]]))

[2, 2, 2] [1, 1, 0]


**Độ phức tạp:** Time `O(n²)`, Space `O(n²)` nếu `zip(*matrix)` được vật chất hóa, thực tế iterator nên `O(n)` thêm.

### Cách 4 — Liệt kê cạnh (tam giác trên) rồi cộng bậc cho hai đầu mút

Với mỗi cặp `i < j` mà `matrix[i][j] == 1` thì có cạnh `i - j`: tăng `deg[i]` **và** `deg[j]` lên 1. Đây là cách xây bậc từ **danh sách cạnh** quen thuộc.

In [11]:
class SolutionEdges:
    def findDegrees(self, matrix):
        n = len(matrix)
        deg = [0] * n
        for i in range(n):
            for j in range(i + 1, n):
                if matrix[i][j] == 1:
                    deg[i] += 1
                    deg[j] += 1
        return deg

print(SolutionEdges().findDegrees([[0, 1, 1], [1, 0, 1], [1, 1, 0]]), SolutionEdges().findDegrees([[0, 1, 0], [1, 0, 0], [0, 0, 0]]))

[2, 2, 2] [1, 1, 0]


**Độ phức tạp:** Time `O(n²/2) = O(n²)`, Space `O(n)`. Chỉ đúng khi ma trận **đối xứng**, đúng như ràng buộc của đề.

### Cách 5 — `map(sum, ...)` kèm kiểm tra bổ đề bắt tay

`list(map(sum, matrix))` rồi (tùy chọn) khẳng định tổng bậc chẵn. Kết hợp kiểm tra tính đúng đắn của đầu vào — tiện khi debug.

In [12]:
class SolutionMapCheck:
    def findDegrees(self, matrix):
        deg = list(map(sum, matrix))
        assert sum(deg) % 2 == 0              # bổ đề bắt tay: tổng bậc = 2E chẵn
        return deg

print(SolutionMapCheck().findDegrees([[0, 1, 1], [1, 0, 1], [1, 1, 0]]), SolutionMapCheck().findDegrees([[0]]))

[2, 2, 2] [0]


**Độ phức tạp:** Time `O(n²)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 4 (liệt kê cạnh) trên ma trận tam giác `[[0,1,1],[1,0,1],[1,1,0]]`:

| (i, j) | `matrix[i][j]` | hành động | deg sau |
|---|---|---|---|
| (0,1) | 1 | `deg[0]++, deg[1]++` | [1,1,0] |
| (0,2) | 1 | `deg[0]++, deg[2]++` | [2,1,1] |
| (1,2) | 1 | `deg[1]++, deg[2]++` | [2,2,2] |

In [13]:
def trace_edges(matrix):
    n = len(matrix)
    deg = [0] * n
    for i in range(n):
        for j in range(i + 1, n):
            if matrix[i][j] == 1:
                deg[i] += 1
                deg[j] += 1
                print(f"cạnh ({i},{j}) -> deg = {deg}")
            else:
                print(f"({i},{j}) không có cạnh")
    return deg

assert trace_edges([[0, 1, 1], [1, 0, 1], [1, 1, 0]]) == [2, 2, 2]
assert trace_edges([[0, 1, 0], [1, 0, 0], [0, 0, 0]]) == [1, 1, 0]

cạnh (0,1) -> deg = [1, 1, 0]
cạnh (0,2) -> deg = [2, 1, 1]
cạnh (1,2) -> deg = [2, 2, 2]
cạnh (0,1) -> deg = [1, 1, 0]
(0,2) không có cạnh
(1,2) không có cạnh


In [14]:
def trace_rows(matrix):
    for i, row in enumerate(matrix):
        print(f"hàng {i}: {row} -> sum = {sum(row)}")
    return [sum(r) for r in matrix]

assert trace_rows([[0]]) == [0]
assert trace_rows([[0, 1, 0], [1, 0, 0], [0, 0, 0]]) == [1, 1, 0]

hàng 0: [0] -> sum = 0
hàng 0: [0, 1, 0] -> sum = 1
hàng 1: [1, 0, 0] -> sum = 1
hàng 2: [0, 0, 0] -> sum = 0


## 9. Code nộp LeetCode

Dùng Cách 2 (tổng từng hàng): ngắn và rõ ràng.

In [15]:
from typing import List, Optional


class Solution:
    def findDegrees(self, matrix: List[List[int]]) -> List[int]:
        return [sum(row) for row in matrix]


print(Solution().findDegrees([[0, 1, 1], [1, 0, 1], [1, 1, 0]]),
      Solution().findDegrees([[0, 1, 0], [1, 0, 0], [0, 0, 0]]),
      Solution().findDegrees([[0]]))

[2, 2, 2] [1, 1, 0] [0]


## 10. Vì sao thuật toán đúng

1. **Định nghĩa bậc.** Bậc của đỉnh `i` là số cạnh nối với `i`. Mỗi cạnh `i - j` ứng với đúng một phần tử `matrix[i][j] = 1` ở hàng `i`.
2. **Không có khuyên.** `matrix[i][i] = 0` nên mỗi cạnh nối `i` với một đỉnh **khác**; không có cạnh tự nối `i` với `i` (nếu có, ta phải cân nhắc đếm nó 1 hay 2 lần).
3. **Mỗi láng giềng đếm đúng một lần.** Hàng `i` có một ô cho mỗi đỉnh `j`, nên mỗi láng giềng góp `1` đúng một lần; không có láng giềng nào bị đếm lặp.
4. **Giá trị chỉ là `0/1`** nên tổng hàng bằng số ô bằng `1`.
5. **Đối xứng** đảm bảo cách tính theo cột (Cách 3) và theo tam giác trên (Cách 4) cho cùng kết quả: cạnh `i - j` hiện ra ở cả `matrix[i][j]` và `matrix[j][i]`.
6. **Bổ đề bắt tay** (`sum(ans) = 2 × số cạnh`) cung cấp phép kiểm tra chéo ở mục 12.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số đỉnh (ma trận có `n²` phần tử).

- Cách 1, 2, 5: `O(n²)` thời gian (phải đọc mọi phần tử), `O(n)` bộ nhớ cho kết quả.
- Cách 3: `O(n²)` thời gian, `O(n)` bộ nhớ thêm nếu `zip` được duyệt dần.
- Cách 4: `O(n²/2)` thời gian (nửa tam giác), `O(n)` bộ nhớ.

Mọi cách đều đạt cận dưới vì phải xem hết `Θ(n²)` phần tử ma trận. Không có best/worst khác nhau (không có thoát sớm).

## 12. Bộ test

Đối chiếu 5 cách + code nộp với ví dụ đề, biên (`n = 1`, đồ thị rỗng, đồ thị đầy đủ) và 300 ma trận ngẫu nhiên **hợp lệ** (đối xứng, đường chéo `0`, giá trị `0/1`, `n <= 100`).

In [16]:
import random

solutions = [SolutionBrute(), SolutionSumRow(), SolutionSumCol(), SolutionEdges(), SolutionMapCheck(), Solution()]

def make_graph(n, p, rng):
    M = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            if rng.random() < p:
                M[i][j] = M[j][i] = 1
    return M

fixed = [
    ([[0, 1, 1], [1, 0, 1], [1, 1, 0]], [2, 2, 2]),
    ([[0, 1, 0], [1, 0, 0], [0, 0, 0]], [1, 1, 0]),
    ([[0]], [0]),
    ([[0, 0], [0, 0]], [0, 0]),
    ([[0, 1], [1, 0]], [1, 1]),
    ([[0, 1, 1, 1], [1, 0, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0]], [3, 1, 1, 1]),      # đồ thị hình sao
    ([[0, 1, 0, 0], [1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0]], [1, 2, 2, 1]),      # đường đi 4 đỉnh
]
for m, expected in fixed:
    for s in solutions:
        assert s.findDegrees([row[:] for row in m]) == expected, (type(s).__name__, m)

rng = random.Random(38980)
for _ in range(300):
    n = rng.randint(1, 100)
    p = rng.choice([0.0, 0.05, 0.3, 0.7, 1.0])
    M = make_graph(n, p, rng)
    outs = [s.findDegrees([row[:] for row in M]) for s in solutions]
    assert all(o == outs[0] for o in outs)
    deg = outs[0]
    assert len(deg) == n and sum(deg) % 2 == 0
    assert sum(deg) == 2 * sum(M[i][j] for i in range(n) for j in range(i + 1, n))      # bổ đề bắt tay
print("tất cả test OK")

tất cả test OK

### Test thuộc tính

(1) Đồ thị đầy đủ `K_n` có mọi bậc bằng `n - 1`; đồ thị rỗng có mọi bậc `0`. (2) Bậc `<= n - 1`. (3) Hoán vị nhãn đỉnh chỉ hoán vị mảng bậc (cùng đa tập hợp bậc).

In [17]:
rng = random.Random(3898)
for n in (1, 2, 5, 30, 100):
    K = [[0 if i == j else 1 for j in range(n)] for i in range(n)]
    assert Solution().findDegrees(K) == [n - 1] * n
    assert Solution().findDegrees([[0] * n for _ in range(n)]) == [0] * n
for _ in range(100):
    n = rng.randint(1, 40)
    M = make_graph(n, 0.4, rng)
    deg = Solution().findDegrees(M)
    assert all(0 <= d <= n - 1 for d in deg)
    perm = list(range(n))
    rng.shuffle(perm)
    P = [[M[perm[i]][perm[j]] for j in range(n)] for i in range(n)]
    assert sorted(Solution().findDegrees(P)) == sorted(deg)
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: cộng cả ma trận thành một số

`sum(map(sum, matrix))` cho **một** số (`2E`), không phải mảng bậc.

In [18]:
def wrong_total(matrix):
    return sum(sum(row) for row in matrix)                        # SAI: một số duy nhất

M = [[0, 1, 0], [1, 0, 0], [0, 0, 0]]
print(wrong_total(M), "!=", Solution().findDegrees(M))
assert wrong_total(M) == 2 and Solution().findDegrees(M) == [1, 1, 0]

2 != [1, 1, 0]


### Lỗi 2: đếm số phần tử khác thay vì cộng (dùng `len`)

`len(row)` luôn bằng `n`, bất kể có cạnh hay không.

In [19]:
def wrong_len(matrix):
    return [len(row) for row in matrix]                           # SAI: luôn là n

M = [[0, 1, 0], [1, 0, 0], [0, 0, 0]]
print(wrong_len(M), "!=", Solution().findDegrees(M))
assert wrong_len(M) == [3, 3, 3] and Solution().findDegrees(M) == [1, 1, 0]

[3, 3, 3] != [1, 1, 0]


### Lỗi 3: duyệt tam giác trên nhưng chỉ cộng cho một đầu mút

Khi dùng `i < j`, mỗi cạnh phải cộng cho **cả hai** đỉnh `i` và `j`. Chỉ cộng cho `i` thì đỉnh có chỉ số lớn bị thiếu bậc.

In [20]:
def wrong_one_endpoint(matrix):
    n = len(matrix)
    deg = [0] * n
    for i in range(n):
        for j in range(i + 1, n):
            if matrix[i][j] == 1:
                deg[i] += 1                                       # SAI: thiếu deg[j] += 1
    return deg

M = [[0, 1, 1], [1, 0, 1], [1, 1, 0]]
print(wrong_one_endpoint(M), "!=", Solution().findDegrees(M))
assert wrong_one_endpoint(M) == [2, 1, 0] and Solution().findDegrees(M) == [2, 2, 2]

[2, 1, 0] != [2, 2, 2]


### Lỗi 4: cộng cả hai đầu mút khi duyệt **toàn bộ** ma trận (đếm gấp đôi)

Duyệt mọi `(i, j)` rồi cộng cho cả `i` và `j` thì mỗi cạnh bị tính 2 lần ở mỗi đầu mút (tổng bậc nhân đôi).

In [21]:
def wrong_double(matrix):
    n = len(matrix)
    deg = [0] * n
    for i in range(n):
        for j in range(n):                                        # SAI: duyệt cả hai tam giác
            if matrix[i][j] == 1:
                deg[i] += 1
                deg[j] += 1
    return deg

M = [[0, 1, 0], [1, 0, 0], [0, 0, 0]]
print(wrong_double(M), "!=", Solution().findDegrees(M))
assert wrong_double(M) == [2, 2, 0] and Solution().findDegrees(M) == [1, 1, 0]

[2, 2, 0] != [1, 1, 0]


### Lỗi 5: dùng `max`/`any` thay vì `sum`

`any(row)` chỉ cho biết đỉnh có ít nhất một cạnh, không cho số cạnh.

In [22]:
def wrong_any(matrix):
    return [int(any(row)) for row in matrix]                      # SAI: chỉ 0 hoặc 1

M = [[0, 1, 1], [1, 0, 1], [1, 1, 0]]
print(wrong_any(M), "!=", Solution().findDegrees(M))
assert wrong_any(M) == [1, 1, 1] and Solution().findDegrees(M) == [2, 2, 2]
assert wrong_any([[0]]) == [0]

[1, 1, 1] != [2, 2, 2]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Bậc của đỉnh từ ma trận kề" = **tổng hàng** (đồ thị vô hướng, không khuyên).
- Từ **danh sách cạnh**: mỗi cạnh `(u, v)` cộng 1 cho cả `u` và `v`. Từ **danh sách kề**: `len(adj[u])`.
- Với đồ thị **có hướng**: tổng hàng = bậc **ra**, tổng cột = bậc **vào** (hai thứ khác nhau).
- Bổ đề bắt tay `sum(deg) = 2E` là phép kiểm tra nhanh kết quả.
- Khi có **khuyên** (`matrix[i][i] = 1`), cần quy ước: thường khuyên cộng `2` vào bậc.

## 15. Biến thể của bài

1. **Find the Town Judge** (997): dùng bậc vào/ra của đồ thị có hướng.
2. **Find Center of Star Graph** (1791): đỉnh có bậc `n - 1`.
3. **Maximal Network Rank** (1615): tổng bậc hai đỉnh trừ cạnh chung.
4. **Convert Adjacency Matrix to Adjacency List** (tự luyện): `[[j for j, v in enumerate(row) if v] for row in M]`.
5. **Count Pairs of Nodes** (1782), **Minimum Degree of a Connected Trio in a Graph** (1761): dùng bậc.
6. Biến thể: đồ thị có hướng (tổng hàng khác tổng cột), đồ thị có trọng số (bậc có trọng số), có khuyên.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai vòng `for` | đếm số `1` mỗi hàng | `O(n²)` | `O(n)` | Dễ hiểu nhất |
| 2. `sum(row)` | tổng từng hàng | `O(n²)` | `O(n)` | Nộp bài |
| 3. `zip(*matrix)` | tổng từng cột (nhờ đối xứng) | `O(n²)` | `O(n)` | Khi cần bậc "vào" |
| 4. Tam giác trên | liệt kê cạnh, cộng 2 đầu | `O(n²/2)` | `O(n)` | Khi cũng cần danh sách cạnh |
| 5. `map` + bắt tay | `map(sum, ...)` + kiểm tra chẵn | `O(n²)` | `O(n)` | Debug/kiểm tra đầu vào |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().findDegrees([[0,1,1,1],[1,0,0,0],[1,0,0,0],[1,0,0,0]])` ra gì?

**Bài 2 (tính tay):** đồ thị có 4 đỉnh và các cạnh `0-1, 1-2, 2-3, 3-0`. Viết ma trận kề và bậc từng đỉnh.

**Bài 3 (điền TODO):** hoàn thành `count_edges(matrix)` trả về số cạnh (dùng bổ đề bắt tay).

**Bài 4 (tự viết lại):** viết `to_adjacency_list(matrix)` trả về danh sách kề `adj[i]` = các đỉnh kề `i`, rồi chứng minh `len(adj[i]) == Solution().findDegrees(matrix)[i]` bằng assert.

**Bài 5 (mini challenge):** viết `max_degree_vertices(matrix)` trả về danh sách các đỉnh có bậc **lớn nhất** (theo thứ tự tăng dần).

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().findDegrees([[0, 1, 1, 1], [1, 0, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0]]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_edges_todo(matrix):
    # TODO: tổng bậc chia 2
    return 0

[3, 1, 1, 1]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: đỉnh 0 nối với ba đỉnh còn lại (hình sao). Bài 2: hình vuông, mỗi đỉnh nối với hai đỉnh kề.

**Mức 2 (vừa):** Bài 3: `sum(map(sum, matrix)) // 2`. Bài 4: `[[j for j, v in enumerate(row) if v == 1] for row in matrix]`. Bài 5: tìm `m = max(deg)` rồi lấy các `i` có `deg[i] == m`.

**Mức 3 (gần đáp án):** Bài 1: `[3, 1, 1, 1]`. Bài 2: ma trận `[[0,1,0,1],[1,0,1,0],[0,1,0,1],[1,0,1,0]]`, bậc `[2,2,2,2]`. Bài 3: tam giác có `3` cạnh.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().findDegrees([[0, 1, 1, 1], [1, 0, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0]]) == [3, 1, 1, 1]
square = [[0, 1, 0, 1], [1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 1, 0]]
assert Solution().findDegrees(square) == [2, 2, 2, 2]

# Bài 3
def count_edges(matrix):
    return sum(map(sum, matrix)) // 2
assert count_edges(square) == 4 and count_edges([[0, 1, 1], [1, 0, 1], [1, 1, 0]]) == 3 and count_edges([[0]]) == 0

# Bài 4
def to_adjacency_list(matrix):
    return [[j for j, v in enumerate(row) if v == 1] for row in matrix]
adj = to_adjacency_list(square)
assert adj == [[1, 3], [0, 2], [1, 3], [0, 2]]
assert all(len(adj[i]) == Solution().findDegrees(square)[i] for i in range(4))

# Bài 5
def max_degree_vertices(matrix):
    deg = Solution().findDegrees(matrix)
    m = max(deg)
    return [i for i, d in enumerate(deg) if d == m]
assert max_degree_vertices([[0, 1, 1, 1], [1, 0, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0]]) == [0]
assert max_degree_vertices(square) == [0, 1, 2, 3] and max_degree_vertices([[0]]) == [0]
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "bậc đỉnh từ ma trận kề":**

```python
return [sum(row) for row in matrix]       # đồ thị vô hướng, không khuyên
# từ danh sách cạnh: deg[u] += 1; deg[v] += 1
```

**Checklist trước khi submit:**

- [ ] Trả về **mảng** `n` phần tử (không phải một tổng duy nhất)?
- [ ] Dùng `sum(row)` (tổng), không phải `len(row)`?
- [ ] Nếu duyệt tam giác trên, đã cộng cho **cả hai** đầu mút của cạnh?
- [ ] Nếu duyệt cả ma trận, chỉ cộng cho **một** đầu mút (nếu không sẽ đếm đôi)?
- [ ] Đã thử `[[0]]`, ma trận toàn 0, đồ thị đầy đủ?
- [ ] Chữ ký đúng `findDegrees(self, matrix: List[List[int]]) -> List[int]`?